# 2. Voter Turnout Heatmap
หน่วยที่ turnout ต่ำ = โอกาสดึงคะแนนในการเลือกตั้งครั้งหน้า

In [3]:
import sys
!{sys.executable} -m pip install folium -q

In [ ]:
import pandas as pd
import folium
from folium.plugins import HeatMap
from pathlib import Path

CLEAN = Path('../cleaned/')
summary = pd.read_csv(CLEAN / 'master_summary_cleaned.csv')
coords  = pd.read_csv(CLEAN / 'coords_cleaned.csv')

JOIN = ['district', 'sub-district', 'unit_number']
s = summary[summary['unit_number'] != -1].copy()
s['turnout'] = (s['valid_ballots'] / s['total_ballots'] * 100).round(1)
s = s.dropna(subset=['turnout'])

df = s.merge(coords, on=JOIN, how='inner')

center = [df['latitude'].mean(), df['longitude'].mean()]

# --- Map 1: Circle marker สีตาม turnout ---
def turnout_color(t):
    if t >= 80:   return '#2ecc71'
    elif t >= 65: return '#f1c40f'
    elif t >= 50: return '#f39c12'
    else:         return '#e74c3c'

m = folium.Map(location=center, zoom_start=9, tiles='CartoDB positron')

for _, row in df.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=7,
        color=turnout_color(row['turnout']),
        fill=True, fill_color=turnout_color(row['turnout']), fill_opacity=0.85,
        popup=folium.Popup(
            f"<b>{row['district']} ต.{row['sub-district']} หน่วย {int(row['unit_number'])}</b><br>"
            f"Turnout: {row['turnout']}%<br>"
            f"ผู้มาใช้สิทธิ์: {int(row['valid_ballots']):,} / {int(row['total_ballots']):,}",
            max_width=250
        ),
        tooltip=f"turnout {row['turnout']}%"
    ).add_to(m)

legend_html = """
<div style="position:fixed;bottom:40px;left:40px;z-index:1000;background:white;
     padding:12px 16px;border-radius:8px;box-shadow:2px 2px 8px rgba(0,0,0,0.3);
     font-family:sans-serif;font-size:13px;">
  <b>Voter Turnout</b><br><br>
  <span style="background:#2ecc71;padding:2px 10px;border-radius:3px;">&nbsp;</span> ≥80% — สูง<br>
  <span style="background:#f1c40f;padding:2px 10px;border-radius:3px;">&nbsp;</span> 65–80% — ปานกลาง<br>
  <span style="background:#f39c12;padding:2px 10px;border-radius:3px;">&nbsp;</span> 50–65% — ต่ำ<br>
  <span style="background:#e74c3c;padding:2px 10px;border-radius:3px;">&nbsp;</span> &lt;50% — ต่ำมาก
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))

print(df[['district','sub-district','unit_number','turnout']].sort_values('turnout').head(10).to_string(index=False))
print(f'\nTurnout เฉลี่ย: {df["turnout"].mean():.1f}%')
print(f'ต่ำสุด: {df["turnout"].min():.1f}%  สูงสุด: {df["turnout"].max():.1f}%')
m


district sub-district  unit_number  turnout
     งาว     บ้านอ้อน            5     39.5
     งาว     บ้านอ้อน            5     40.5
วังเหนือ     ร่องเคาะ           13     40.8
วังเหนือ     ร่องเคาะ           12     41.4
วังเหนือ     ร่องเคาะ           13     43.9
วังเหนือ     ร่องเคาะ           12     44.8
     งาว        ปงเตา            7     45.3
     งาว        ปงเตา            7     45.8
     งาว     บ้านร้อง            9     47.5
     งาว        ปงเตา           11     48.1

Turnout เฉลี่ย: 63.2%
ต่ำสุด: 39.5%  สูงสุด: 81.1%


: 